In [2]:
import os
import ast
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb

# 1. Setup Base Data and Columns (Match this to your actual dataframe)
df = pd.read_csv('data/spectral_feature_data.csv')
base_spectral_columns = ['410', '435', '460', '485', ...]
target_cols = ['p2.N.low', 'p2.OC', 'p2.P', ...]


non_feature_cols = [col for col in df.columns if col.startswith('p4')]
feature_cols = [col for col in df.columns if col not in non_feature_cols]

base_spectral_columns = [col for col in feature_cols if not col.startswith("p")]
target_cols = [col for col in df.columns if col.startswith("p")]


config_map = {
    "Spectral Only": [False, False],
    "Spectral + pH": [True, False],
    "Spectral + EC": [False, True],
    "Spectral + pH + EC": [True, True]
}

# 2. Setup the Folder Structure
base_dir = "models"
model_types = ["plsr", "xgb", "lgbm", "rf"]

for m_type in model_types:
    os.makedirs(os.path.join(base_dir, m_type), exist_ok=True)
    print(f"Directory ready: {os.path.join(base_dir, m_type)}/")


print(target_cols, base_spectral_columns)

Directory ready: models\plsr/
Directory ready: models\xgb/
Directory ready: models\lgbm/
Directory ready: models\rf/
['p1.pH.index', 'p1.EC.ds_m', 'p1.Clay.wt_pct', 'p1.Sand.wt_pct', 'p1.Silt.wt_pct', 'p2.N.wt_pct', 'p2.Zn.mg_kg', 'p2.OC.wt_pct', 'p3.Fe.mg_kg', 'p3.K.mg_kg', 'p3.P.mg_kg', 'p3.S.wt_pct', 'p4.BD.g_cm3', 'p4.CEC.cmolc_kg', 'p4.CF.wt_pct', 'p4.WR_10kPa.wt_pct', 'p4.WR_1500kPa.wt_pct', 'p4.WR_33kPa.wt_pct'] ['410', '435', '460', '485', '510', '535', '560', '585', '610', '645', '680', '705', '730', '760', '810', '860', '900', '940']


In [3]:
# 3. Load your Winning Blueprints (Assuming you saved the best configs per model to CSVs)
# Note: Ensure you have these CSVs ready from your previous analysis steps.
blueprints = {
    "plsr": pd.read_csv("results/model_configs/PLSR_final_configs.csv"),
    "xgb": pd.read_csv("results/model_configs/XGBoost_final_configs.csv"),
    "lgbm": pd.read_csv("results/model_configs/LGBM_final_configs.csv"),
    "rf": pd.read_csv("results/model_configs/RF_final_configs.csv")
}


In [4]:

# 4. Master Loop: Train and Save
for m_type in model_types:
    print(f"\n========== BUILDING {m_type.upper()} MODELS ==========")
    blueprint_df = blueprints[m_type]
    
    for _, row in blueprint_df.iterrows():
        
        target = row['Feature']

        if m_type == "xgb":
            best_config = "Spectral + pH + EC"
        else:
            best_config = row['Config']
        

        # --- NEW: Conditional Parameter Parsing ---
        if m_type == "plsr":
            # PLSR only saved the integer under 'Best_Components'
            n_comp = int(row['Best_Components'])
            best_params = {'n_components': n_comp}
        else:
            # XGB, LGBM, and RF saved dictionary strings under 'Best_Params'
            best_params = ast.literal_eval(row['Best_Params']) if pd.notna(row['Best_Params']) else {}

        
        print(f"Training {m_type.upper()} for {target} using {best_config}...")

        # --- Reconstruct the exact dataset for this configuration ---
        prediction_columns = base_spectral_columns.copy()
        maskpH = pd.Series(True, index=df.index)
        maskEC = pd.Series(True, index=df.index)

        if config_map[best_config][0]: # pH is True
            prediction_columns.append("p1.pH.index")
            maskpH = df["p1.pH.index"].notna()
        if config_map[best_config][1]: # EC is True
            prediction_columns.append("p1.EC.ds_m")
            maskEC = df["p1.EC.ds_m"].notna()

        feature_mask = maskpH & maskEC
        X_custom = df[prediction_columns]
        
        target_mask = df[target].notna()
        final_mask = target_mask & feature_mask 
        
        y_clean = df.loc[final_mask, target]
        X_clean = X_custom.loc[final_mask]


        # --- NEW: DEBUGGING & SAFETY VALVE ---
        '''print(f"  -> Total rows in df: {len(df)}")
        print(f"  -> Rows with {target} (not NaN): {target_mask.sum()}")
        print(f"  -> Rows with pH (not NaN): {maskpH.sum()}")
        print(f"  -> Rows with EC (not NaN): {maskEC.sum()}")'''
        print(f"  -> Rows surviving all filters: {final_mask.sum()}")
        
        if final_mask.sum() == 0:
            print(f"⚠️ SKIPPING {m_type.upper()} for {target} - No data left after filtering!")
            continue

        # Use the EXACT same split so the Stacker can use the unseen X_test later
        X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

        # --- Initialize the specific Algorithm ---
        if m_type == "plsr":
            # PLSR expects 'n_components'
            engine = PLSRegression(**best_params)
        elif m_type == "xgb":
            engine = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, **best_params)
        elif m_type == "lgbm":
            engine = LGBMRegressor(random_state=42, verbose=-1, **best_params)
        elif m_type == "rf":
            engine = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)

        # --- Create the Production Pipeline ---
        # 1. Impute missing values automatically
        # 2. Run the algorithm
        model_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('model', engine)
        ])

        # --- Train on the Training Data ---
        # We wrap X_train in a DataFrame to prevent the feature name warnings
        X_train_df = pd.DataFrame(X_train, columns=prediction_columns)
        model_pipeline.fit(X_train_df, y_train)

        # --- Save the Pipeline ---
        # File name format: models/rf/p2.N.low_model.pkl
        file_path = os.path.join(base_dir, m_type, f"{target}_model.pkl")
        
        # We also save the 'prediction_columns' list so your main.py knows EXACTLY 
        # which columns this specific model expects to receive from the hardware.
        save_package = {
            'features_required': prediction_columns,
            'pipeline': model_pipeline
        }
        
        joblib.dump(save_package, file_path)
        print(f"  -> Saved to {file_path}")

print("\nAll winning models have been successfully serialized for production.")


========== BUILDING PLSR MODELS ==========
Training PLSR for p1.Clay.wt_pct using Spectral Only...
  -> Rows surviving all filters: 3915
  -> Saved to models\plsr\p1.Clay.wt_pct_model.pkl
Training PLSR for p1.EC.ds_m using Spectral + pH + EC...
  -> Rows surviving all filters: 21832
  -> Saved to models\plsr\p1.EC.ds_m_model.pkl
Training PLSR for p1.Sand.wt_pct using Spectral + pH + EC...
  -> Rows surviving all filters: 4306
  -> Saved to models\plsr\p1.Sand.wt_pct_model.pkl
Training PLSR for p1.Silt.wt_pct using Spectral + pH...
  -> Rows surviving all filters: 3755
  -> Saved to models\plsr\p1.Silt.wt_pct_model.pkl
Training PLSR for p1.pH.index using Spectral + pH + EC...
  -> Rows surviving all filters: 21832
  -> Saved to models\plsr\p1.pH.index_model.pkl
Training PLSR for p2.N.wt_pct using Spectral + pH + EC...
  -> Rows surviving all filters: 21832
  -> Saved to models\plsr\p2.N.wt_pct_model.pkl
Training PLSR for p2.OC.wt_pct using Spectral + pH + EC...
  -> Rows surviving all 

In [6]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# --- Configuration ---
# Make sure these match your previous script exactly
model_types = ["plsr", "xgb", "lgbm", "rf"]
# base_dir = "models" 
eval_dir = "evaluations"
os.makedirs(eval_dir, exist_ok=True)

# Helper function for Mean Percentage Error (MPE)
def mean_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Prevent division by zero
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return 0.0 # Fallback if all true values are exactly 0
    return np.mean((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100

# Get all unique targets across your blueprints
all_targets = set()
for m_type in model_types:
    all_targets.update(blueprints[m_type]['Feature'].unique())

print("Starting Evaluation Pipeline...")

for target in all_targets:
    print(f"\n========== EVALUATING {target} ==========")
    
    # ---------------------------------------------------------
    # 1. RECONSTRUCT THE DATASET
    # ---------------------------------------------------------
    # We need to apply the same masking to get the exact same split.
    # We will assume the most complex config (Spectral + pH + EC) 
    # was used to ensure all models have the necessary columns available.
    
    prediction_columns = base_spectral_columns.copy() + ["p1.pH.index", "p1.EC.ds_m"]
    maskpH = df["p1.pH.index"].notna()
    maskEC = df["p1.EC.ds_m"].notna()
    target_mask = df[target].notna()
    
    final_mask = target_mask & maskpH & maskEC
    
    if final_mask.sum() == 0:
        print(f"⚠️ Skipping {target} - Not enough overlapping data.")
        continue
        
    y_clean = df.loc[final_mask, target]
    X_clean = df.loc[final_mask, prediction_columns]
    
    # EXACT same split
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )
    
    # ---------------------------------------------------------
    # 2. LOAD MODELS & GENERATE PREDICTIONS
    # ---------------------------------------------------------
    test_predictions = {}
    train_cv_predictions = {}
    
    for m_type in model_types:
        file_path = os.path.join(base_dir, m_type, f"{target}_model.pkl")
        
        if not os.path.exists(file_path):
            print(f"  Missing {m_type.upper()} model for {target}, skipping this model.")
            continue
            
        # Load the saved pipeline package
        save_package = joblib.load(file_path)
        pipeline = save_package['pipeline']
        required_features = save_package['features_required']
        
        # Subset the X data to only what this specific model requires
        X_train_model = X_train[required_features]
        X_test_model = X_test[required_features]
        
        # Predict on Test Set (for final evaluation)
        test_predictions[m_type.upper()] = pipeline.predict(X_test_model)
        
        # Generate Cross-Validated Predictions on Train Set (for the Stacker)
        # cv=5 means 5-fold cross-validation
        train_cv_predictions[m_type.upper()] = cross_val_predict(
            pipeline, X_train_model, y_train, cv=5, n_jobs=-1
        )
    
    if not test_predictions:
        print(f"  No models found for {target}. Moving to next.")
        continue

    # ---------------------------------------------------------
    # 3. TRAIN THE STACKER (META-MODEL)
    # ---------------------------------------------------------
    print("  Training Linear Stacker...")
    # Convert prediction dictionaries to 2D numpy arrays (Features for the meta-model)
    X_train_meta = pd.DataFrame(train_cv_predictions)
    X_test_meta = pd.DataFrame(test_predictions)
    
    # Train Linear Regression on the unbiased CV predictions
    stacker = LinearRegression()
    stacker.fit(X_train_meta, y_train)
    
    # Get final stacked predictions for the test set
    test_predictions['STACKER'] = stacker.predict(X_test_meta)
    
    # ---------------------------------------------------------
    # 4. CALCULATE METRICS
    # ---------------------------------------------------------
    models_to_eval = list(test_predictions.keys())
    metrics_data = []
    
    for model_name in models_to_eval:
        preds = test_predictions[model_name]
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        mpe = mean_percentage_error(y_test, preds)
        
        metrics_data.append({
            'Model': model_name,
            'R2': r2,
            'MAE': mae,
            'MPE': mpe
        })
        
    metrics_df = pd.DataFrame(metrics_data)
    
    # ---------------------------------------------------------
    # 5. GENERATE DASHBOARD
    # ---------------------------------------------------------
    print("  Generating Dashboard...")
    sns.set_theme(style="whitegrid")
    # Create a 2-row layout: Top row for scatters, bottom row for metrics
    fig = plt.figure(figsize=(24, 12))
    fig.suptitle(f'Model Evaluation Dashboard: {target}', fontsize=24, fontweight='bold', y=0.98)
    
    # Subplot grid: 2 rows. Top row has 5 columns, Bottom row has 3 columns.
    gs = fig.add_gridspec(2, 15) 
    
    # --- TOP ROW: Scatter Plots ---
    for i, model_name in enumerate(models_to_eval):
        ax = fig.add_subplot(gs[0, i*3:(i+1)*3])
        preds = test_predictions[model_name]
        
        # Actual vs Predicted Scatter
        ax.scatter(y_test, preds, alpha=0.6, edgecolors='w', s=50)
        
        # 1:1 Perfect Prediction Line
        min_val = min(y_test.min(), preds.min())
        max_val = max(y_test.max(), preds.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Line')
        
        # Linear Trendline (np.polyfit)
        z = np.polyfit(y_test, preds, 1)
        p = np.poly1d(z)
        ax.plot(y_test, p(y_test), 'r-', lw=2, label='Trendline')
        
        ax.set_title(model_name, fontsize=16, fontweight='bold')
        ax.set_xlabel('Actual Values', fontsize=12)
        ax.set_ylabel('Predicted Values', fontsize=12)
        ax.legend()

    # --- BOTTOM ROW: Bar Charts ---
    # R-Squared
    ax_r2 = fig.add_subplot(gs[1, 1:5])
    sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
    ax_r2.set_title('$R^2$ Score (Higher is Better)', fontsize=14)
    ax_r2.set_ylim(0, 1.0) # Standard R2 range
    
    # MAE
    ax_mae = fig.add_subplot(gs[1, 6:10])
    sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
    ax_mae.set_title('Mean Absolute Error (Lower is Better)', fontsize=14)
    
    # MPE
    ax_mpe = fig.add_subplot(gs[1, 11:15])
    sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')
    ax_mpe.set_title('Mean Percentage Error (%)', fontsize=14)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95]) # Adjust to fit main title
    
    # Save Figure
    save_path = os.path.join(eval_dir, f"{target}_dashboard.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  -> Dashboard saved to {save_path}")

print("\nAll evaluations complete!")

Starting Evaluation Pipeline...

========== EVALUATING p4.BD.g_cm3 ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  Training Linear Stacker...
  Generating Dashboard...


C:\Users\jagda\AppData\Local\Temp\ipykernel_46324\621144416.py:172: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_46324\621144416.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_46324\621144416.py:183: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')


  -> Dashboard saved to evaluations\p4.BD.g_cm3_dashboard.png

========== EVALUATING p4.WR_10kPa.wt_pct ==========
⚠️ Skipping p4.WR_10kPa.wt_pct - Not enough overlapping data.

========== EVALUATING p1.Sand.wt_pct ==========


...

  -> Dashboard saved to evaluations\p1.Silt.wt_pct_dashboard.png

========== EVALUATING p3.P.mg_kg ==========


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  Training Linear Stacker...
  Generating Dashboard...


C:\Users\jagda\AppData\Local\Temp\ipykernel_46324\621144416.py:172: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
C:\Users\jagda\AppData\Local\Temp\ipykernel_46324\621144416.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
C:\Users\jagda\AppData\Local\Temp\ipykernel_46324\621144416.py:183: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')


  -> Dashboard saved to evaluations\p3.P.mg_kg_dashboard.png

All evaluations complete!
